# 🎲 Monte Carlo & Retirement Planning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/03_monte_carlo.ipynb)

All four PV return models (historical bootstrap, statistical, parameterized normal & fat-tailed Student-t) **plus block bootstrap**, and all five withdrawal rules (fixed real, fixed %, smoothed %, RMD-style, custom schedule).

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Simulation settings {display-mode: "form"}
tickers = "VTI, BND"          #@param {type:"string"}
weights = "60, 40"            #@param {type:"string"}
history_start = "2008-01-01"  #@param {type:"date"}
initial_balance = 1000000     #@param {type:"number"}
years = 30                    #@param {type:"slider", min:5, max:60, step:5}
simulations = 5000            #@param {type:"number"}
return_model = "block_bootstrap"  #@param ["bootstrap", "block_bootstrap", "normal", "student_t", "statistical"]
withdrawal_rule = "fixed"     #@param ["none", "fixed", "fixed_pct", "smoothed_pct", "rmd", "custom"]
annual_withdrawal = 40000     #@param {type:"number"}
withdrawal_percent = 0.04     #@param {type:"number"}
current_age = 60              #@param {type:"number"}
life_expectancy = 92          #@param {type:"number"}
annual_inflation = 0.025      #@param {type:"number"}
SMOKE = False

In [ ]:
import pandas as pd
from portlab import monte_carlo, plots
from portlab.data import get_returns
from portlab.optimize import mean_cov

tick_list = [t.strip().upper() for t in tickers.split(",") if t.strip()]
w = pd.Series([float(x) for x in weights.split(",")], index=tick_list)
w = w / w.sum()
rets = get_returns(tick_list, history_start)
port_monthly = ((1 + (rets * w.values).sum(axis=1)).resample("ME").prod() - 1)
mu, cov = mean_cov(rets)

res = monte_carlo(
    initial=initial_balance, years=years,
    n_sims=simulations if not SMOKE else 200,
    model=return_model, hist_returns=port_monthly,
    mean_annual=float(mu @ w), vol_annual=float((w @ cov @ w) ** 0.5),
    asset_mu=mu, asset_cov=cov, weights=w,
    withdrawal=withdrawal_rule, withdrawal_amount=annual_withdrawal,
    withdrawal_pct=withdrawal_percent, current_age=current_age,
    life_expectancy=life_expectancy, inflation_annual=annual_inflation)

print(f"SUCCESS RATE: {res.success_rate:.1%}")
res.ending_stats().map(lambda v: f"{v:,.2f}" if abs(v) > 10 else f"{v:.2%}")

In [ ]:
plots.fan_chart(res.percentiles((5, 10, 25, 50, 75, 90, 95))).show()

In [ ]:
# Withdrawal-rule shoot-out — which policy survives?
import pandas as pd
rules = {"4% fixed real": dict(withdrawal="fixed", withdrawal_amount=0.04*initial_balance),
         "4% of balance": dict(withdrawal="fixed_pct", withdrawal_pct=0.04),
         "Smoothed 4%": dict(withdrawal="smoothed_pct", withdrawal_pct=0.04),
         "RMD-style": dict(withdrawal="rmd")}
rows = {}
for name, kw in rules.items():
    r = monte_carlo(initial=initial_balance, years=years,
                    n_sims=2000 if not SMOKE else 200, model=return_model,
                    hist_returns=port_monthly, current_age=current_age,
                    life_expectancy=life_expectancy,
                    inflation_annual=annual_inflation, **kw)
    rows[name] = {"Success": r.success_rate,
                  "Median End": float(pd.Series(r.balances[:, -1]).median()),
                  "P10 End": float(pd.Series(r.balances[:, -1]).quantile(0.1))}
pd.DataFrame(rows).T.style.format({"Success": "{:.1%}", "Median End": "{:,.0f}", "P10 End": "{:,.0f}"})

**Financial goals / asset-liability modeling (PV's paid planners):** model any life plan with `withdrawal_rule='custom'` — build a `custom_schedule` series of monthly amounts (+contributions while working, −tuition, −house down-payment, −retirement spending) and the success rate tells you if the plan survives. Import a schedule from CSV with `pd.read_csv(...)["amount"]`.